In [1]:
import pandas as pd
import numpy as np
from plz_to_bundesland import get_bundesland

car_data = pd.read_excel('../data/raw/fz10_2026_09.xlsx', sheet_name='Sheet1')

car_data = car_data.drop(index=0).reset_index(drop=True)



car_data = car_data[
    (~car_data['Marke'].str.strip().str.endswith('ZUSAMMEN', na=False)) &
    (~car_data['Marke'].str.strip().str.endswith('INSGESAMT', na=False))
].copy()

car_data['Marke'] = car_data['Marke'].ffill()


car_data = car_data.replace('-', 0)

column_mapping = {
    'Marke' : 'brand',
    'Modellreihe': 'model',
    'Insgesamt': 'total_sep_2026',
    'Unnamed: 3': 'total_jan_sep_2026',
    'Unnamed: 4': 'total_share_pct',
    'mit Dieselantrieb': 'diesel_sep_2026',
    "Unnamed: 6": 'diesel_jan_sep_2026',
    'Unnamed: 7': 'diesel_share_pct',
    "mit Hybridantrieb \n(incl. Plug-in-Hybrid)": 'hybrid_incl_phev_sep_2026',
    'Unnamed: 9': 'hybrid_incl_phev_jan_sep_2026',
    'Unnamed: 10': 'hybrid_incl_phev_share_pct',
    'Benzin-Hybridantrieb \n(incl. Plug-in-Hybrid)': 'petrol_hybrid_incl_phev_sep_2026',
    'Unnamed: 12': 'petrol_hybrid_incl_phev_jan_sep_2026',
    'Unnamed: 13': 'petrol_hybrid_incl_phev_share_pct',
    'Diesel-Hybridantrieb \n(incl. Plug-in-Hybrid)': 'diesel_hybrid_incl_phev_sep_2026',
    'Unnamed: 15': 'diesel_hybrid_incl_phev_jan_sep_2026',
    'Unnamed: 16': 'diesel_hybrid_incl_phev_share_pct',
    'Hybridantrieb \n(ohne Plug-in-Hybrid)': 'hybrid_non_plugin_sep_2026',
    'Unnamed: 18': 'hybrid_non_plugin_jan_sep_2026',
    'Unnamed: 19': 'hybrid_non_plugin_share_pct',
    'Benzin-Hybridantrieb \n(ohne Plug-in-Hybrid)': 'petrol_hybrid_non_plugin_sep_2026',
    'Unnamed: 21': 'petrol_hybrid_non_plugin_jan_sep_2026',
    'Unnamed: 22': 'petrol_hybrid_non_plugin_share_pct',
    'Diesel-Hybridantrieb \n(ohne Plug-in-Hybrid)': 'diesel_hybrid_non_plugin_sep_2026',
    'Unnamed: 24': 'diesel_hybrid_non_plugin_jan_sep_2026',
    'Unnamed: 25': 'diesel_hybrid_non_plugin_share_pct',
    'Plug-in-Hybridantrieb': 'phev_sep_2026',
    'Unnamed: 27': 'phev_jan_sep_2026',
    'Unnamed: 28': 'phev_share_pct',
    'Benzin-Plug-in-Hybridantrieb': 'petrol_phev_sep_2026',
    'Unnamed: 30': 'petrol_phev_jan_sep_2026',
    'Unnamed: 31': 'petrol_phev_share_pct',
    'Diesel-Plug-in-Hybridantrieb': 'diesel_phev_sep_2026',
    'Unnamed: 33': 'diesel_phev_jan_sep_2026',
    'Unnamed: 34': 'diesel_phev_share_pct',
    'mit Elektroantrieb (BEV)': 'bev_sep_2026',
    'Unnamed: 36': 'bev_jan_sep_2026',
    'Unnamed: 37': 'bev_share_pct',
    'mit Allradantrieb': 'all_wheel_drive_sep_2026',
    'Unnamed: 39': 'all_wheel_drive_jan_sep_2026',
    'Unnamed: 40': 'all_wheel_drive_share_pct',
    'Cabriolets': 'cabriolets_sep_2026',
    'Unnamed: 42': 'cabriolets_jan_sep_2026',
    'Unnamed: 43': 'cabriolets_share_pct'

}

car_data = car_data.rename(columns=column_mapping, errors='raise')


numeric_cols = car_data.columns.drop(['brand', 'model'])

car_data[numeric_cols] = car_data[numeric_cols].apply(
    pd.to_numeric,
    errors='raise'
)


car_data[car_data['brand'] == 'BMW'].filter(regex='^.*_pct$')

df = car_data.filter(regex=r'^(?!.*_jan_sep_).*$')


df_subset = df[
    (df['brand'] == 'BMW') &
    (
        (df['bev_sep_2026'] > 0) |
        (df['phev_sep_2026'] > 0)
    )
].copy()

df_subset['chargeable_vehicles'] = (
    df_subset['bev_sep_2026'] +
    df_subset['phev_sep_2026']
)

df_subset = df_subset[
    ['brand', 'model', 'bev_sep_2026',
     'phev_sep_2026', 'chargeable_vehicles']
]

print(df_subset.sort_values(by='chargeable_vehicles', ascending=False))


# fuel_cols = [col for col in car_data.columns if col.startswith('ZS')]
# print(*fuel_cols)

# bmw_data = car_data[car_data['Marke'] == "BMW"][['Berichtsjahr', 'Marke', 'Modellreihe', 'Anzahl']]

# print(len(bmw_data))
# print(bmw_data.sort_values(by='Anzahl', ascending=False))

   brand     model  bev_sep_2026  phev_sep_2026  chargeable_vehicles
44   BMW        X1          1809            343                 2152
46   BMW        X3          1072            825                 1897
39   BMW       5ER           523            509                 1032
38   BMW       4ER           777              0                  777
48   BMW        X5             0            370                  370
37   BMW       3ER           140            193                  333
43   BMW        IX           218              0                  218
45   BMW        X2           136              0                  136
40   BMW       7ER            76             13                   89
51   BMW        XM             0             64                   64
36   BMW       2ER             0             32                   32
54   BMW  SONSTIGE             0              2                    2


In [2]:
charging_stations = pd.read_csv('../data/raw/rhein-kreis-neuss-ladesaulen-in-deutschland.csv', sep=';')

In [3]:
charging_stations['Postleitzahl_original'] = (
    charging_stations['Postleitzahl'].copy()
)

In [4]:
plz = charging_stations['Postleitzahl'].astype('string').str.zfill(5)

charging_stations['Bundesland'] = plz.map(
    lambda x: get_bundesland(x) if pd.notna(x) else 'Unknown'
)

charging_stations['Bundesland'] = (
    charging_stations['Bundesland'].replace('Unknown', pd.NA)
)

In [6]:
known = charging_stations.dropna(subset=['Bundesland'])

valid_cities = (
    known.groupby('Ort')['Bundesland']
    .nunique()
    .loc[lambda x: x == 1]
    .index
)

city_lookup = (
    known[known['Ort'].isin(valid_cities)]
    .drop_duplicates('Ort')
    .set_index('Ort')['Bundesland']
)

charging_stations['Bundesland'] = (
    charging_stations['Bundesland'].fillna(
        charging_stations['Ort'].map(city_lookup)
    )
)

In [8]:
known = charging_stations.dropna(subset=['Bundesland'])

valid_districts = (
    known.groupby('Kreis/kreisfreie Stadt')['Bundesland']
    .nunique()
    .loc[lambda x: x == 1]
    .index
)

district_lookup = (
    known[known['Kreis/kreisfreie Stadt'].isin(valid_districts)]
    .drop_duplicates('Kreis/kreisfreie Stadt')
    .set_index('Kreis/kreisfreie Stadt')['Bundesland']
)

charging_stations['Bundesland'] = (
    charging_stations['Bundesland'].fillna(
        charging_stations['Kreis/kreisfreie Stadt'].map(district_lookup)
    )
)

In [9]:
remaining = charging_stations[
    charging_stations['Bundesland'].isna()
]

print("Unresolved rows:", len(remaining))

display(
    remaining[
        ['Ort', 'Postleitzahl', 'Kreis/kreisfreie Stadt',
         'Straße', 'Hausnummer', 'Bundesland']
    ]
)

Unresolved rows: 1


,Ort,Postleitzahl,Kreis/kreisfreie Stadt,Straße,Hausnummer,Bundesland
59532,Weimar,9925,Kreisfreie Stadt Weimar,Henry-van-de-Velde-Straße,2,<NA>


In [10]:
charging_stations.at[59532,'Postleitzahl'] = 99425
charging_stations.at[59532,'Bundesland'] = "Thüringen"

In [11]:
expected = charging_stations['Ort'].map(city_lookup)

mismatches = charging_stations[
    expected.notna()
    & charging_stations['Bundesland'].notna()
    & charging_stations['Bundesland'].ne(expected).fillna(False)
]

print(mismatches[
        ['Ort', 'Postleitzahl', 'Bundesland',
         'Kreis/kreisfreie Stadt', 'Straße', 'Hausnummer']
    ])


Empty DataFrame
Columns: [Ort, Postleitzahl, Bundesland, Kreis/kreisfreie Stadt, Straße, Hausnummer]
Index: []


In [12]:
print("Missing Bundesland:",
      charging_stations['Bundesland'].isna().sum())

print("Remaining city/state mismatches:",
      len(mismatches))

print("Total records:", len(charging_stations))

Missing Bundesland: 0
Remaining city/state mismatches: 0
Total records: 61793


In [13]:
plz = charging_stations['Postleitzahl'].astype('string')

print("Missing PLZ:", plz.isna().sum())
print("Not five digits:",
      (~plz.str.fullmatch(r'\d{5}').fillna(False)).sum())

Missing PLZ: 0
Not five digits: 3588


In [164]:


charging_stations['Postleitzahl'] = charging_stations['Postleitzahl'].astype(str)
charging_stations.at[1162,'Postleitzahl'] = "06571"
charging_stations.at[1166,'Postleitzahl'] = "07607"
charging_stations.at[1167,'Postleitzahl'] = "07607"
charging_stations.at[1168,'Postleitzahl'] = "07613"
charging_stations.at[1169,'Postleitzahl'] = "07639"

# cities = [
#     'Leipzig', 'Dresden', 'Chemnitz', 'Gera',
#     'Pirna', 'Jena', 'Kamenz', 'Weißenfels',
#     'Freiberg', 'Görlitz', 'Plauen', 'Hoyerswerda',
#     'Sangerhausen', 'Limbach-Oberfrohna', 'Cottbus',
#     'Markkleeberg', 'Dessau-Roßlau', 'Halle (Saale)',
#     'Halle', 'Bitterfeld-Wolfen', 'Zwickau'
# ]

bundeslaender = [
    'Sachsen', 'Sachsen-Anhalt', 'Thüringen', 'Brandenburg'
]

mask = (
    (
        # charging_stations['Ort'].isin(cities) |
        charging_stations['Bundesland'].isin(bundeslaender)
    )
    & (charging_stations['Postleitzahl'].str.len() == 4)
)

bayern_mask = (
    (charging_stations['Bundesland'] == 'Bayern') &
    (charging_stations['Postleitzahl'].str.len() == 4)
)


charging_stations.loc[mask, 'Postleitzahl'] = (
    charging_stations.loc[mask, 'Postleitzahl'].str.zfill(5)
)

charging_stations.loc[bayern_mask, 'Postleitzahl'] = (
    '9' + charging_stations.loc[bayern_mask, 'Postleitzahl']
)

charging_stations.loc[52639, 'Postleitzahl'] = "33102"
charging_stations.loc[28265, 'Postleitzahl'] = "50933"
charging_stations.loc[9035, 'Postleitzahl'] = "53919"
charging_stations.loc[59644, 'Postleitzahl'] = "53919"
charging_stations.loc[37483, 'Postleitzahl'] = "72072"
charging_stations.loc[39172, 'Postleitzahl'] = "72793"
charging_stations.loc[43320, 'Postleitzahl'] = "55444"
charging_stations.loc[30173, 'Postleitzahl'] = "23558"

print(len(charging_stations[charging_stations['Postleitzahl'].astype(str).str.len() < 5]))

# print(
#     (~charging_stations['Postleitzahl']
#       .str.fullmatch(r'\d{5}')).sum()
# )

0


In [166]:
display(
    charging_stations[(charging_stations['Bundesland'] == 'Schleswig-Holstein') & (charging_stations['Postleitzahl'].str.len() < 5)]
)

,Betreiber,Art der Ladeeinrichung,Anzahl Ladepunkte,Steckertypen1,Steckertypen2,Steckertypen3,Steckertypen4,P1 [kW],P2 [kW],P3 [kW],P4 [kW],Nennleistung Ladeeinrichtung [kW],Kreis/kreisfreie Stadt,Ort,Postleitzahl,Straße,Hausnummer,Adresszusatz,Inbetriebnahmedatum,koordinaten,Anzeigename (Karte),Steckertypen5,P5 [kW],Public Key5,Steckertypen6,P6 [kW],Public Key6,Postleitzahl_original,Bundesland


In [165]:
mask = charging_stations['Postleitzahl'].str.len() < 5

charging_stations[mask].groupby('Bundesland').size().sort_values(
    ascending=False
)

Series([], dtype: int64)

In [114]:
charging_stations['Ort'] = charging_stations['Ort'].replace(
    'Halle', 'Halle (Saale)'
)

In [167]:
print("Invalid PLZ format:",
      (~charging_stations['Postleitzahl'].str.fullmatch(r'\d{5}')).sum())

print("Missing Bundesland:",
      charging_stations['Bundesland'].isna().sum())

print("Total records:", len(charging_stations))

Invalid PLZ format: 0
Missing Bundesland: 0
Total records: 61793


In [169]:
charging_stations.to_csv(
    '../data/raw/charging_stations_cleaned.csv',
    index=False
)